# Semana 15: Auditoría ética de un modelo

**Introducción a la Ciencia de Datos** · Dudbil Olvasada Pabón Riaño · UNAB

Cuaderno guiado. Cada celda corresponde a un ejercicio de la guía de laboratorio; los resultados deben coincidir con sus puntos de control. Los datos se leen desde el repositorio del curso.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
BASE = "https://raw.githubusercontent.com/dudbilopr/Introduccion-a-la-ciencia-de-datos/main/Semanas/Datos/"

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
a = pd.read_csv(BASE + "censo_adult.csv")
a["y"] = (a["income"] == ">50K").astype(int)
print(a.shape, " vacíos como '?':", {c: int(v) for c, v in (a == "?").sum().items() if v})

## 1. El espejo

In [ ]:
print(a.groupby("sex")["y"].agg(["count", "mean"]).round(3))
print(a.groupby("race")["y"].agg(["count", "mean"]).round(3))

## 2. Modelo sin la columna sexo

In [ ]:
num = ["age", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
cat = ["workclass", "marital_status", "occupation", "relationship", "race", "native_country"]
X = pd.get_dummies(a[num + cat], columns=cat, drop_first=True).astype(float)
Xtr, Xte, ytr, yte, str_, ste = train_test_split(X, a["y"], a["sex"], test_size=0.3, random_state=42, stratify=a["y"])
m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(Xtr, ytr)
p = m.predict(Xte)
print("Prueba:", len(yte), " exactitud", round((p == yte).mean(), 3), " línea base", round(1 - yte.mean(), 3))

## 3. Errores por grupo

In [ ]:
for g in ["Female", "Male"]:
    k = (ste == g).values
    tn, fp, fn, tp = confusion_matrix(yte[k], p[k]).ravel()
    print(g, k.sum(), " real", round(yte[k].mean(), 3), " predicha", round(p[k].mean(), 3), " sensibilidad", round(tp / (tp + fn), 3),
          " falsos positivos", round(fp / (fp + tn), 3), " precisión", round(tp / (tp + fp), 3))

## 4. Variables sustitutas

In [ ]:
print(pd.crosstab(a["relationship"], a["sex"]))
X2 = X[[c for c in X.columns if not c.startswith(("relationship_", "marital_status_"))]]
m2 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(X2.loc[Xtr.index], ytr)
p2 = m2.predict(X2.loc[Xte.index])
print("Exactitud", round((p2 == yte).mean(), 3), {g: round(p2[(ste == g).values].mean(), 3) for g in ["Female", "Male"]})

## 5. Reidentificación

In [ ]:
for cols in [["age", "sex", "race"], ["age", "sex", "race", "native_country"],
             ["age", "sex", "race", "native_country", "occupation"], ["age", "sex", "race", "native_country", "occupation", "education"]]:
    unicas = (a.groupby(cols).size() == 1).sum()
    print(len(cols), "atributos:", unicas, "personas únicas,", round(unicas / len(a) * 100, 1), "%")

## Declaración de uso de IA

Si usó una herramienta de IA, escriba aquí: cuál, para qué, el prompt, la versión del modelo y qué verificó usted.